# Prompt 12 — Choose and evaluate a defensible contribution
**Study:** Token-Level Uncertainty as a Predictor of Hallucination in Large Language Models.  
**Pathway:** B — controlled comparative study.  
**Research question:** Is 90th-percentile token NLL a better ranking signal for hallucination than mean token NLL?  
**Rule:** No test-set performance metrics are calculated in this notebook. The test set is reserved for Prompt 13. Results must come from real execution.

## Three pathways considered
| Path | Scientific question | Code / compute | Benefit | Risk / evidence |
|---|---|---|---|---|
| A. Reproduction + extension | Does an existing method transfer to a justified new condition? | New condition and likely new scoring; moderate compute | External validity | Needs independently justified condition and faithful reference implementation |
| B. Controlled comparison **(selected)** | Do mean NLL and p90 NLL differ in hallucination ranking performance? | Reuse scores; low compute | Isolates aggregation choice | Limited novelty; validate score direction and use matched examples |
| C. Targeted modification | Does a changed aggregation rule improve detection? | Isolated scoring or aggregation change; low–moderate compute | Clear ablation | Tuning risk and need for theoretical rationale |

**Controls:** Same DistilGPT-2 model, dataset, labels, splits, tokenization and NLL scores. Answer length is a negative/control comparator. **Primary metric:** Average Precision (AP), larger is better. **Secondary:** AUROC. Use validation for descriptive method selection only; do not tune on test.

In [ ]:
import io, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from google.colab import files
print('Upload AITR_Prompt10_FINAL.zip')
uploaded = files.upload()
assert len(uploaded)==1, 'Upload exactly one ZIP'
zip_name = next(iter(uploaded))
assert zip_name.lower().endswith('.zip')
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    required = ['train_scores.csv','validation_scores.csv','test_scores.csv','baseline_validation.json']
    assert set(required).issubset(z.namelist()), 'Missing expected ZIP entries'
    frames = {s:pd.read_csv(io.BytesIO(z.read(f'{s}_scores.csv'))) for s in ['train','validation','test']}
    audit = json.loads(z.read('baseline_validation.json'))
print('Baseline checks:', audit['checks'])

In [ ]:
# Verify the integrity of the imported score files without inspecting test performance.
for split,df in frames.items():
    assert df.shape[0] == {'train':420,'validation':90,'test':90}[split]
    assert df['ID'].is_unique and df['status'].eq('ok').all()
    assert set(df['label'].unique()) == {0,1}
    assert df[['mean_nll','p90_nll','answer_token_length']].notna().all().all()
    assert np.isfinite(df[['mean_nll','p90_nll','answer_token_length']].to_numpy()).all()
    assert (df[['mean_nll','p90_nll','answer_token_length']] >= 0).all().all()
assert pd.concat([x['ID'] for x in frames.values()]).is_unique
print('=== PROMPT 12 INPUT INTEGRITY: PASS ===')

## Validation-only comparison
Higher NLL is assumed to indicate greater uncertainty; verify that assumption descriptively. **AP** measures ranking quality for hallucination (label 1) and depends on prevalence. **AUROC** is a secondary ranking metric. Answer length is a control, not an uncertainty measure. These are validation-set results, **not final test results**.

In [ ]:
val = frames['validation'].copy()
y = val['label'].astype(int).to_numpy()
comparators = {'mean_nll':'Mean token NLL (baseline)',
               'p90_nll':'90th percentile token NLL (proposed)',
               'answer_token_length':'Answer length (control)'}
rows=[]
for column,name in comparators.items():
    score = val[column].astype(float).to_numpy()
    rows.append({'method':name,'column':column,'n':len(y),
                 'positives':int(y.sum()),'average_precision':float(average_precision_score(y,score)),
                 'auroc':float(roc_auc_score(y,score))})
results = pd.DataFrame(rows)
print('=== VALIDATION COMPARISON (NOT TEST) ===')
print(results.to_string(index=False, float_format=lambda v:f'{v:.5f}'))

## Prespecified choice and deeper analysis
The primary scientific comparison is **p90 NLL versus mean NLL**, regardless of which performs better. For an optional downstream single-signal choice, select the higher validation AP, breaking exact ties in favor of mean NLL (simpler baseline). This is a **validation-only** choice. Perform answer-length sensitivity by computing validation AP within shorter/longer answer groups; report low sample sizes. No significance claims from this small sample.

In [ ]:
mean_ap = float(results.loc[results.column=='mean_nll','average_precision'].iloc[0])
p90_ap = float(results.loc[results.column=='p90_nll','average_precision'].iloc[0])
selected = 'p90_nll' if p90_ap > mean_ap else 'mean_nll'
print('Frozen single-signal choice (validation AP):',selected)
median_len = float(val['answer_token_length'].median())
subrows=[]
for group,subset in [('short',val[val.answer_token_length<=median_len]),
                     ('long',val[val.answer_token_length>median_len])]:
    for col in ['mean_nll','p90_nll']:
        ap = float(average_precision_score(subset.label,subset[col])) if subset.label.nunique()==2 else np.nan
        subrows.append({'group':group,'method':col,'n':len(subset),
                        'positives':int(subset.label.sum()),'validation_ap':ap})
sensitivity = pd.DataFrame(subrows)
print('=== LENGTH SENSITIVITY (VALIDATION ONLY) ===')
print(sensitivity.to_string(index=False))

In [ ]:
# Freeze the decision and export evidence. Do NOT open the test scores for performance here.
out=Path('/content/AITR_Prompt12_outputs');out.mkdir(parents=True,exist_ok=True)
results.to_csv(out/'validation_comparison.csv',index=False)
sensitivity.to_csv(out/'validation_length_sensitivity.csv',index=False)
protocol = {
 'prompt':12,'study_pathway':'B - controlled comparative study',
 'research_question':'Does p90 token NLL rank hallucinations better than mean token NLL?',
 'model':'distilgpt2 surrogate scorer','source_zip':zip_name,
 'splits':{'train':420,'validation':90,'test':90},
 'positive_label':1,'score_direction':'higher means more predicted hallucination',
 'primary_metric':'average_precision','secondary_metric':'auroc',
 'comparators':['mean_nll','p90_nll','answer_token_length'],
 'primary_contrast':'p90_nll versus mean_nll',
 'single_signal_selection':'higher validation AP; exact tie -> mean_nll',
 'selected_single_signal':selected,
 'test_set_status':'NOT EVALUATED; reserved for Prompt 13',
 'original_row_audit':audit['checks'].get('original_row_audit'),
 'limitations':['Small validation set (90 rows; 14 positive cases)',
                'Synthetic split-specific IDs, not original HaluEval source IDs',
                'DistilGPT-2 is a surrogate, not generator confidence',
                'Answer-length sensitivity is exploratory']}
(out/'frozen_comparison_protocol.json').write_text(json.dumps(protocol,indent=2))
with zipfile.ZipFile('/content/AITR_Prompt12_outputs.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in out.iterdir():z.write(f,arcname=f.name)
print('=== PROMPT 12 COMPLETE (VALIDATION ONLY) ===')
print('Selected:',selected)
print('Saved: /content/AITR_Prompt12_outputs.zip')
files.download('/content/AITR_Prompt12_outputs.zip')